# Module 01: Inference APIs

[Workshop home](../README.md) · [Setup](../docs/00_prework.md) · [Endpoint reference](../docs/inference.md) · [Troubleshooting](../docs/troubleshooting.md)

**25 minutes.** Your first request to Akamai Cloud Inference, then the rest of what
you need to use it properly.

Akamai Cloud Inference speaks the OpenAI API. The useful consequence is that code you
have already written works here after changing two values. You will see that swap
happen in Step 3.

Before you start, `python scripts/check_setup.py` should pass.

| Step | What |
|---|---|
| 1 | Connect |
| 2 | Send one raw HTTP request and read what comes back |
| 3 | The same call through the OpenAI SDK |
| 4 | Stream, and measure time to first token |
| 5 | Control the output: temperature, seed, stop, max_tokens |
| 6 | See what is served and choose a model |
| 7 | Break things on purpose and read the errors |

## Step 1: Connect

Two values do all the work: the base URL and your key. Both come from `.env`, so no
key is ever typed into a notebook cell.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# Works whether you started Jupyter in the repo root or in this module folder.
root = Path.cwd()
if not (root / ".env").is_file() and (root.parent / ".env").is_file():
    root = root.parent
if not (root / ".env").is_file():
    raise RuntimeError(
        "No .env found. Run 'cp .env.example .env' and add your key. See docs/00_prework.md"
    )

load_dotenv(root / ".env", override=True)
BASE_URL = os.environ["AKAMAI_BASE_URL"].strip().rstrip("/")
API_KEY = os.environ["AKAMAI_API_KEY"].strip()

if not BASE_URL.endswith("/v1"):
    raise RuntimeError(f"AKAMAI_BASE_URL is {BASE_URL!r}. It must end in /v1.")
if not API_KEY or API_KEY == "paste_your_key_here":
    raise RuntimeError("AKAMAI_API_KEY is not set. See docs/get-an-api-key.md")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=60.0)
MODEL = "qwen3-8b"

print("Endpoint:", BASE_URL)
print("Key loaded. Value hidden.")
print("Model:", MODEL)

## Step 2: One raw HTTP request

Before any SDK hides it, send the request yourself. A chat completion is a `POST` to
`/chat/completions` with a bearer token and a JSON body. That is the whole protocol.

Watch three things in the output: the status code, the `x-ratelimit` headers that
tell you your own limits, and the shape of the JSON.

In [ ]:
import json
import requests

http = requests.post(
    f"{BASE_URL}/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
    json={
        "model": MODEL,
        "messages": [{"role": "user", "content": "Say hello in one short sentence."}],
        "max_tokens": 30,
    },
    timeout=60,
)

print("Status:", http.status_code)

limits = {k: v for k, v in http.headers.items() if k.lower().startswith("x-ratelimit")}
print("Rate limit headers:", limits or "none returned on this account")
print()
print(json.dumps(http.json(), indent=2)[:800])

Three fields matter in that response.

| Field | Why |
|---|---|
| `choices[0].message.content` | The answer |
| `choices[0].finish_reason` | Whether the answer is complete. `stop` is good, `length` means it was cut off |
| `usage` | Tokens in and out, which is what you are billed on |

Read the rate limit headers rather than assuming a number. They depend on your
account, which is why this workshop prints them instead of documenting them.

Everything from here uses the SDK, which sends exactly this request.

## Step 3: The same call through the SDK

This is the part worth paying attention to. The OpenAI Python client takes a
`base_url`. Point it at Akamai, pass your key, and every OpenAI example you have ever
read works unchanged.

```python
client = OpenAI(base_url="https://api.akamai-inference.com/v1", api_key=...)
```

That is the whole migration.

In [ ]:
reply = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are concise. Answer in two sentences."},
        {"role": "user", "content": "What is an API key?"},
    ],
    max_tokens=120,
    temperature=0.0,
)

print(reply.choices[0].message.content)
print()
print("finish_reason:", reply.choices[0].finish_reason)
print("tokens in/out:", reply.usage.prompt_tokens, "/", reply.usage.completion_tokens)

`messages` is the entire conversation, and the endpoint keeps no memory of it. Your
application stores the history and resends what it wants the model to see. The
`system` message sets behaviour; `user` is the person talking.

**Your turn.** Change the `user` message to a question about something you are
actually building, and run the cell again.

## Step 4: Streaming

A non-streamed call returns nothing until the whole answer exists. Streaming returns
tokens as they are produced, so the user sees text almost immediately. The total time
is the same. The *perceived* time is not, which is why almost every chat interface
streams.

Time to first token, or TTFT, is the number that matters for how responsive something
feels.

In [ ]:
import time

start = time.monotonic()
ttft = None
usage = None

stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Name three uses for text embeddings, one short line each."}],
    max_tokens=150,
    temperature=0.0,
    stream=True,
    stream_options={"include_usage": True},
)

for chunk in stream:
    if chunk.usage:
        usage = chunk.usage
    if chunk.choices and chunk.choices[0].delta.content:
        if ttft is None:
            ttft = time.monotonic() - start
        print(chunk.choices[0].delta.content, end="", flush=True)

elapsed = time.monotonic() - start
print()
print(f"\ntime to first token : {ttft:.2f}s")
print(f"total               : {elapsed:.2f}s")
if usage:
    print(f"output tokens       : {usage.completion_tokens}")
    print(f"tokens per second   : {usage.completion_tokens / max(elapsed - (ttft or 0), 1e-6):.1f}")

Some chunks carry no text. They hold the role, the finish reason, or the usage block
at the end of the stream, which is why the loop checks before printing.

`stream_options={"include_usage": True}` is what makes the final usage chunk appear.
Without it you have to estimate output tokens with a local tokenizer, and that
estimate will not match what you are billed.

## Step 5: Controlling the output

Four parameters cover almost everything you will need.

`temperature` controls randomness. Use `0.0` when you want the same answer every
time, such as classification or extraction. Raise it when you want variety.

In [ ]:
def ask(prompt, **kwargs):
    r = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}], **kwargs
    )
    return (r.choices[0].message.content or "").strip()

prompt = "Invent a name for a coffee shop. Reply with the name and nothing else."

print("temperature = 0.0")
for _ in range(2):
    print("   ", ask(prompt, max_tokens=20, temperature=0.0))

print("\ntemperature = 1.3")
for _ in range(2):
    print("   ", ask(prompt, max_tokens=20, temperature=1.3))

`stop` halts generation when a string appears. `max_tokens` caps the length. Both
are worth knowing because both can silently truncate an answer.

In [ ]:
# stop: generation halts at the string, and the string itself is not returned.
print("with stop=['4'] ->", repr(ask("Count up: 1, 2, 3, 4, 5", max_tokens=40,
                                     temperature=0.0, stop=["4"])))

# max_tokens: the model is cut off mid sentence if the budget runs out.
short = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain how TLS works, in detail."}],
    max_tokens=25,
)
print("\ntruncated      ->", repr(short.choices[0].message.content))
print("finish_reason  ->", short.choices[0].finish_reason)

**The rule to take away.** Check `finish_reason` before you use a response.

| Value | Meaning |
|---|---|
| `stop` | Finished normally. Safe to use |
| `length` | Cut off at `max_tokens`. The content is incomplete |
| `tool_calls` | The model wants you to run a function. `content` is empty on purpose |

This matters most when you parse the output. A truncated JSON response is not valid
JSON, and `json.loads` will fail on it. Module 03 comes back to this.

## Step 6: What is served, and which to use

Ask the endpoint rather than guessing a model name. Then pay attention to what it
does **not** tell you, because that shapes how you write against it.

In [ ]:
raw = requests.get(
    f"{BASE_URL}/models", headers={"Authorization": f"Bearer {API_KEY}"}, timeout=30
).json()

print("Models served:")
for m in sorted(raw["data"], key=lambda m: m["id"]):
    print("   ", m["id"])

print("\nFields returned for each model:")
print("   ", sorted(raw["data"][0].keys()))

Four ids and almost nothing else. No context window, no list of supported features,
no hint about which model accepts images.

So limits have to come from somewhere else. The reliable way to find a context window
is to deliberately ask for too much and read the number out of the error, which is
exactly what Step 7 does next. These are the limits the endpoint enforced when this
workshop was written:

| Model | Enforced context limit | Use it for |
|---|---|---|
| `qwen3-8b` | 32,768 | Chat, tool calling, reasoning. The default here |
| `qwen3-4b` | 32,768 | The same work, smaller and cheaper |
| `gemma-4-26b-a4b-it` | 262,143 | Image input, and long documents |
| `qwen3-embedding-4b` | 8,192 | Embeddings, in module 02 |

Treat that table as a starting point and confirm it yourself. An earlier version of
this endpoint did report `max_model_len` on `/v1/models` and has since stopped, so
anything written down about a live service has a shelf life.

One warning that will save you time later: use `qwen3-8b` for anything involving
tools. `gemma-4-26b-a4b-it` is the largest model here and the worst at tool calling.
Module 04 covers why.

## Step 7: Break it on purpose

Error messages on this endpoint are specific and quote the numbers you need. Reading
them now means recognising them later.

In [ ]:
def show_error(label, call):
    try:
        call()
        print(f"{label:<18} no error raised")
    except Exception as exc:
        print(f"{label:<18} {type(exc).__name__}")
        print(f"{'':<18} {str(exc)[:150]}")

show_error("bad model name", lambda: client.chat.completions.create(
    model="not-a-real-model", messages=[{"role": "user", "content": "hi"}], max_tokens=5))

show_error("empty messages", lambda: client.chat.completions.create(
    model=MODEL, messages=[], max_tokens=5))

show_error("prompt too long", lambda: client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "word " * 40000}], max_tokens=5))

| Code | Cause | Fix |
|---|---|---|
| `401` | Key wrong, or has stray whitespace | Check `.env` |
| `404` | Model name typo, **or** a base URL missing `/v1` | List the models; check the URL |
| `422` | Malformed body or a field of the wrong type | Read the message, it names the field |
| `400` | Understood but unservable, usually too long | Shorten the input |
| `429` | Rate limited | Back off, retry a bounded number of times |

Two things worth knowing.

**That 404.** A base URL without the `/v1` suffix returns `404` on every request and
looks exactly like a bad model name. It is the most common setup mistake here.

**The word "Estimated".** The too-long error reads `Estimated input token count
(50000) exceeds the model limit (32768)`. The endpoint estimates the count itself,
and the estimate will not match your local tokenizer. A prompt you measured as
fitting can still be rejected, so leave headroom instead of aiming at the limit.

## Validation

Confirms the things this module claimed to teach actually happened.

In [ ]:
served = {m.id for m in client.models.list().data}
assert MODEL in served, f"{MODEL} is not served. Pick one of: {sorted(served)}"
assert reply.choices[0].message.content, "the chat completion returned no content"
assert ttft is not None and ttft > 0, "streaming produced no text chunks"
assert short.choices[0].finish_reason == "length", "max_tokens did not truncate"
print("Module 01 complete.")
print("You can call the endpoint, stream, control the output, and read its errors.")

## Continue

Nothing was created, so there is nothing to clean up.

Next: **[Module 02: Embeddings and RAG](../02_Embeddings_and_RAG/)**, where the model
starts answering from your documents instead of from memory.

Reference: [endpoint details](../docs/inference.md) · [troubleshooting](../docs/troubleshooting.md)